In [ ]:
import json
import os

from dotenv import load_dotenv
import openai

load_dotenv()

MODEL = "gpt-4o-mini"

SYSTEM_PROMPT = """
    You are a Movie Expert agent. Choose the correct tool for each user question.

    Tools:
    - get_popular_movies — list popular movies (GET /movies). Use for questions about what is popular or trending now.
    - get_movie_details — details for one movie by numeric id (GET /movies/:id). Use when the user asks what a movie is by ID.
    - get_movie_credits — cast and crew (GET /movies/:id/credits). Use when the user asks who stars, who worked on, or credits for a movie ID.

    When the user wants movie data, respond with exactly one tool call. Parse movie IDs from the user text as integers.
    """

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Fetch popular movies from the /movies endpoint.",
            "parameters": {
                "type": "object",
                "properties": {},
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Fetch movie information for a given movie id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "Movie id."},
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Fetch cast and crew for a given movie id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "Movie id."},
                },
                "required": ["id"],
            },
        },
    },
]

def print_tool_calls(response: openai.types.chat.chat_completion.ChatCompletion) -> None:
    msg = response.choices[0].message
    if not msg.tool_calls:
        print("No tool_calls (model returned text).")
        print(msg.content)
        return
    for tc in msg.tool_calls:
        name = tc.function.name
        args = tc.function.arguments
        print(f"function: {name}")
        print(f"arguments: {args}")

def request_tool_calls(user_content: str) -> openai.types.chat.chat_completion.ChatCompletion:
    client = openai.OpenAI()
    return client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ],
        tools=TOOLS,
        tool_choice="auto",
    )

In [ ]:
TEST_PROMPTS = [
    "지금 인기 있는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화가 무엇인지 알려줘",
    "movie ID 550에 해당하는 영화에 누가 출연하는지 알려줘",
]

for i, prompt in enumerate(TEST_PROMPTS, start=1):
    print(f"\n--- Test {i} ---")
    print(f"user: {prompt}")
    r = request_tool_calls(prompt)
    print_tool_calls(r)